### Import Llibraries

In [14]:
import os
import numpy as np
import pandas as pd 
import matplotlib as plt
import mne

In [15]:
root = "../data"

### Subject selection and feature definition

In [16]:
SUBJECTS = ['S001']

In [17]:
VALID_RUNS = [
    "R04", # Left/Right fist imagery
    "R08", 
    "R12"
]

MOTOR_CHANNELS = [
    'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.',
    'C3..', 'C1..', 'Cz..', 'C2..', 'C4..',
    'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.'
]

### EDF Loader + Filtering

In [18]:
def load_and_preprocess(edf_path):

    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose=False
    )

    available_channels = [
        ch for ch in MOTOR_CHANNELS
        if ch in raw.ch_names
    ]

    raw.pick(available_channels)

    # Filter for Mu and Beta rhythms (motor imagery)
    raw.filter(
        l_freq=8,
        h_freq=30,
        verbose=False
    )

    return raw

### Epoch extractor

In [19]:
def extract_epochs(edf_path):

    raw = load_and_preprocess(edf_path)

    events, event_dict = mne.events_from_annotations(raw, verbose=False)

    # Safely extract ONLY T1 (Left) and T2 (Right) events
    # This automatically drops T0 (Rest) without hardcoding ID numbers
    target_events = {
        k: v for k, v in event_dict.items() 
        if k in ['T1', 'T2']
    }

    epochs = mne.Epochs(
        raw,
        events,
        event_id=target_events,
        tmin=1,
        tmax=4,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data(copy=True)
    y = epochs.events[:, 2]

    return X, y

### Create dataset

In [20]:
def build_dataset():

    X_all = []
    y_all = []

    for subject in SUBJECTS:
        print(f"\nProcessing {subject}")

        subject_path = os.path.join(root, subject)
        
        if not os.path.exists(subject_path):
            print(f"Path not found: {subject_path}")
            continue

        for file in sorted(os.listdir(subject_path)):

            if not file.endswith(".edf"):
                continue

            run_name = file.split(".")[0][-3:]

            if run_name not in VALID_RUNS:
                continue

            edf_path = os.path.join(subject_path, file)

            try:
                X, y = extract_epochs(edf_path)
                X_all.extend(X)
                y_all.extend(y)
                print(f"{run_name}: {len(y)} epochs extracted")

            except Exception as e:
                print(f"Failed: {file}")
                print(e)

    return np.array(X_all), np.array(y_all)

In [21]:
edf_path = os.path.join(root, SUBJECTS[0], "S001R04.edf")

raw = load_and_preprocess(edf_path)
epochs = extract_epochs(edf_path)

X, Y = build_dataset()


Processing S001
R04: 15 epochs extracted
R08: 15 epochs extracted
R12: 15 epochs extracted


In [22]:
raw

<RawEDF | S001R04.edf, 15 x 20000 (125.0 s), ~2.3 MiB, data loaded>

In [23]:
epochs

(array([[[ 8.92598487e-06,  1.14981795e-06, -5.14823953e-06, ...,
           4.04601195e-06,  1.26712583e-05,  1.42189039e-05],
         [ 1.86668682e-05,  1.17164709e-05,  5.07079479e-07, ...,
           1.45175828e-06,  9.36474071e-06,  1.44408965e-05],
         [ 2.13316858e-05,  1.82077960e-05,  7.29454262e-06, ...,
           9.34015932e-07,  6.42026325e-06,  1.08902793e-05],
         ...,
         [ 2.48109955e-05,  2.01707226e-05,  7.09643269e-06, ...,
           1.38728751e-05,  1.89028068e-05,  1.98675973e-05],
         [ 2.29930128e-05,  2.21297561e-05,  9.12416708e-06, ...,
           1.41150375e-05,  1.68991450e-05,  1.56712139e-05],
         [ 1.83210258e-05,  1.98678806e-05,  9.95186813e-06, ...,
           1.87199465e-05,  1.79525232e-05,  1.22494527e-05]],
 
        [[ 1.09950525e-06, -1.34998685e-06, -6.11856163e-06, ...,
          -2.01773421e-05, -1.42447349e-05, -7.39970189e-06],
         [ 5.64130507e-06,  1.88239343e-06, -6.47604610e-06, ...,
          -1.77600066

In [25]:
X

array([[[ 8.92598487e-06,  1.14981795e-06, -5.14823953e-06, ...,
          4.04601195e-06,  1.26712583e-05,  1.42189039e-05],
        [ 1.86668682e-05,  1.17164709e-05,  5.07079479e-07, ...,
          1.45175828e-06,  9.36474071e-06,  1.44408965e-05],
        [ 2.13316858e-05,  1.82077960e-05,  7.29454262e-06, ...,
          9.34015932e-07,  6.42026325e-06,  1.08902793e-05],
        ...,
        [ 2.48109955e-05,  2.01707226e-05,  7.09643269e-06, ...,
          1.38728751e-05,  1.89028068e-05,  1.98675973e-05],
        [ 2.29930128e-05,  2.21297561e-05,  9.12416708e-06, ...,
          1.41150375e-05,  1.68991450e-05,  1.56712139e-05],
        [ 1.83210258e-05,  1.98678806e-05,  9.95186813e-06, ...,
          1.87199465e-05,  1.79525232e-05,  1.22494527e-05]],

       [[ 1.09950525e-06, -1.34998685e-06, -6.11856163e-06, ...,
         -2.01773421e-05, -1.42447349e-05, -7.39970189e-06],
        [ 5.64130507e-06,  1.88239343e-06, -6.47604610e-06, ...,
         -1.77600066e-05, -5.14453435e

In [26]:
Y

array([3, 2, 2, 3, 3, 2, 3, 2, 3, 2, 2, 3, 2, 3, 2, 2, 3, 2, 3, 2, 3, 2,
       3, 3, 2, 2, 3, 3, 2, 2, 3, 2, 3, 2, 2, 3, 3, 2, 2, 3, 3, 2, 3, 2,
       3])